# 🚀 QUANTUM ANALYTICS GROUP — Guion Completo
## Antigravity (VS Code + Anaconda Python 3.12) — Mac
### EXPLO-RA 2026 · Énfasis II · Unicomfacauca · Docente: Edward Zúñiga Dorado

---

### Cómo usar este notebook
- Ejecuta las celdas **en orden**, de arriba hacia abajo (`Shift + Enter`)
- Cada celda tiene título, contexto y el código listo para correr
- El dashboard se lanza **desde la terminal de VS Code** (Parte 5)
- En caso de falla del dashboard: usa las celdas de la **Parte 4** como respaldo

### Estructura
| Parte | Contenido | Celdas |
|---|---|---|
| **1** | Verificar librerías y archivos | 1 |
| **2** | Limpieza del S03 — 6 pasos | 6 |
| **3** | Verificación del número canónico | 1 |
| **4** | Análisis de respaldo — una celda por pregunta | 6 |
| **5** | Crear y lanzar el dashboard | 2 |
| **6** | Mapa de navegación para el EXPLO-RA | — |

> **Número que nunca puede fallar:** Leticia = −$79,342 / −50.4%  
> Si no aparece este número en la Parte 3, hay un error en la limpieza.


---
## PARTE 1 — Verificar librerías y archivos

> Entorno confirmado: **Anaconda Python 3.12.4** — todas las librerías ya están instaladas.  
> Esta celda solo verifica que todo esté disponible y que los archivos de datos estén en la carpeta correcta.  
> Si algún archivo falta: arrastrarlo a la carpeta del proyecto en VS Code.


In [1]:
# Traemos las herramientas: os (sistema), pandas (datos), plotly (gráficos), streamlit (web)
import os, pandas as pd, plotly, streamlit as st, openpyxl

# ── Librerías ──────────────────────────────────────────────────────────────
print("LIBRERÍAS DISPONIBLES")
print(f"  pandas     {pd.__version__}")
print(f"  plotly     {plotly.__version__}")
print(f"  streamlit  {st.__version__}")
print(f"  openpyxl   {openpyxl.__version__}")
print()

# ── Archivos de datos ──────────────────────────────────────────────────────
ARCHIVOS = [
    'S03_Ventas_Datos_Sucios_v4.xlsx'
]
print("ARCHIVOS EN LA CARPETA DEL PROYECTO")
for f in ARCHIVOS:
    estado = "✅" if os.path.exists(f) else "❌  FALTA — arrastrar a esta carpeta en VS Code"
    print(f"  {estado}  {f}")

falta = [f for f in ARCHIVOS if not os.path.exists(f)]
print()
if falta:
    print("❌  Faltan archivos. Agrégalos antes de continuar.")
else:
    print("✅  Todo listo — puedes continuar con la Parte 2.")

LIBRERÍAS DISPONIBLES
  pandas     2.2.2
  plotly     5.22.0
  streamlit  1.32.0
  openpyxl   3.1.2

ARCHIVOS EN LA CARPETA DEL PROYECTO
  ✅  S03_Ventas_Datos_Sucios_v3.xlsx
  ❌  FALTA — arrastrar a esta carpeta en VS Code  NovaMarket_S01_Dataset_v2.csv

❌  Faltan archivos. Agrégalos antes de continuar.


---
## PARTE 2 — Limpieza de Datos (6 pasos)

> El archivo `S03_Ventas_Datos_Sucios_v4.xlsx` tiene **512 registros con 4 tipos de errores**.  
> Al completar los 6 pasos obtendrás **exactamente 500 registros limpios** — el dataset canónico del curso.  
> Cada paso incluye una verificación automática con `assert` — si falla, aparece el error antes de avanzar.


### Paso 1 — Cargar el archivo sucio y diagnóstico inicial

In [2]:
import pandas as pd
import numpy as np

# header=2: las 2 primeras filas son decorativas, los encabezados reales están en la fila 3
# pd.read_excel lee el archivo de Excel. Lo guardamos en una variable llamada 'df_raw' (Dataframe Crudo)
df_raw = pd.read_excel('S03_Ventas_Datos_Sucios_v4.xlsx'
                       sheet_name='Ventas_Datos_Sucios',
                       header=2)

print("=" * 52)
print("  DIAGNÓSTICO — ARCHIVO SUCIO (antes de limpiar)")
print("=" * 52)
# len() cuenta cuántas filas tiene nuestra tabla actualmente
print(f"  Total filas:               {len(df_raw)}")
print(f"  Duplicados exactos:        {df_raw.duplicated().sum()}")
print(f"  Nulos en Cantidad:         {df_raw['Cantidad'].isnull().sum()}")
print(f"  Ciudades únicas:           {df_raw['Ciudad'].nunique()}")
print(f"  Categorías únicas:         {df_raw['Categoria'].nunique()}")
print("=" * 52)
print(f"  OBJETIVO: {len(df_raw)} filas → 500 filas limpias")

  DIAGNÓSTICO — ARCHIVO SUCIO (antes de limpiar)
  Total filas:               512
  Duplicados exactos:        12
  Nulos en Cantidad:         5
  Ciudades únicas:           16
  Categorías únicas:         12
  OBJETIVO: 512 filas → 500 filas limpias


### Paso 2 — Eliminar los 12 duplicados exactos

In [3]:
# ELIMINAR DUPLICADOS: drop_duplicates() borra filas idénticas.
# reset_index(drop=True) reorganiza la numeración de las filas del 0 al 499.
# ELIMINAR DUPLICADOS: drop_duplicates() borra filas idénticas.
# reset_index(drop=True) reorganiza la numeración de las filas del 0 al 499.
df = df_raw.drop_duplicates().reset_index(drop=True)

print(f"Antes: {len(df_raw)} filas")
print(f"Después: {len(df)} filas")
print(f"Eliminados: {len(df_raw) - len(df)} duplicados")

assert len(df) == 500, f"ERROR: esperaba 500 filas, hay {len(df)}"
print("✅ 500 registros — correcto")

Antes: 512 filas
Después: 500 filas
Eliminados: 12 duplicados
✅ 500 registros — correcto


### Paso 3 — Estandarizar ciudades

In [4]:
# Creamos una lista con los nombres oficiales que aceptamos
CIUDADES_VALIDAS = ['Bogotá','Medellín','Cali','Barranquilla','Cartagena','Leticia']

ciudad_map = {
    'bogota':'Bogotá',   'BOGOTÁ':'Bogotá',   'Bogota ':'Bogotá', 'Bogota':'Bogotá',
    'barranquilla':'Barranquilla', 'Barranqilla':'Barranquilla',
    'BARRANQUILLA':'Barranquilla',
    'cartagena':'Cartagena', 'CARTAGENA':'Cartagena', 'Cartajena':'Cartagena',
    'leticia':'Leticia', 'LETICIA':'Leticia',
    'medellin':'Medellín', 'Medellin':'Medellín',
    'cali':'Cali',       'CALI':'Cali',
}

# ESTANDARIZAR TEXTO: str.strip() elimina espacios al inicio y final.
# replace(ciudad_map) busca los nombres malos y los cambia por los buenos según el diccionario.
# ESTANDARIZAR TEXTO: str.strip() elimina espacios. replace(ciudad_map) cambia lo malo por lo bueno.
df['Ciudad'] = df['Ciudad'].str.strip().replace(ciudad_map)

# Verificamos si quedó alguna ciudad que NO esté (~) en nuestra lista de ciudades válidas
invalidas = df[~df['Ciudad'].isin(CIUDADES_VALIDAS)]['Ciudad'].unique()
assert len(invalidas) == 0, f"Quedan ciudades inválidas: {invalidas}"
print(f"✅ Ciudades válidas: {sorted(df['Ciudad'].unique())}")

✅ Ciudades válidas: ['Barranquilla', 'Bogotá', 'Cali', 'Cartagena', 'Leticia', 'Medellín']


### Paso 4 — Estandarizar categorías

In [5]:
CATEGORIAS_VALIDAS = ['Laptops','Smartphones','Audio','Wearables']

cat_map = {
    'laptops':'Laptops', 'LAPTOPS':'Laptops', 'Laptop':'Laptops',
    'smartphones':'Smartphones', 'SMARTPHONES':'Smartphones',
    'SmartPhones':'Smartphones',
    'audio':'Audio',   'AUDIO':'Audio',
    'wearables':'Wearables', 'WEARABLES':'Wearables', 'Wereables':'Wearables',
}

# str.strip() quita espacios, replace(cat_map) aplica el diccionario para corregir nombres.
# Aplicamos la misma lógica: quitamos espacios y usamos el diccionario cat_map para corregir.
df['Categoria'] = df['Categoria'].str.strip().replace(cat_map)

invalidas = df[~df['Categoria'].isin(CATEGORIAS_VALIDAS)]['Categoria'].unique()
assert len(invalidas) == 0, f"Quedan categorías inválidas: {invalidas}"
print(f"✅ Categorías válidas: {sorted(df['Categoria'].unique())}")

✅ Categorías válidas: ['Audio', 'Laptops', 'Smartphones', 'Wearables']


### Paso 5 — Imputar nulos en Cantidad

> **Criterio oficial:** mediana por categoría. No se elimina la fila — el resto de columnas tiene información válida.
>
> | Categoría | Mediana | IDs afectados |
> |---|---|---|
> | Audio | 3 | 1244 (Cali) |
> | Laptops | 2 | 1115 (Cali) |
> | Smartphones | 2 | 1450 (Barranquilla) |
> | Wearables | 3 | 1319 (Bogotá), 1008 (Barranquilla) |
>
> ⚠️ **Caso especial ID 1008** (Smartwatch Q, Barranquilla): la mediana imputa **3**, pero el valor real canónico es **1**.  
> Con envío de $220 en Barranquilla el impacto en utilidad total es de ~$360 — marginal y aceptable.  
> En Leticia ($1,650 envío) el mismo error sería crítico.


In [6]:
# Mediana por categoría — calculada sobre los registros con Cantidad válida
# CALCULAR MEDIANA: groupby('Categoria') agrupa los datos por tipo de producto.
# median() calcula el valor central estadístico de la cantidad vendida para cada grupo.
# CALCULAR MEDIANA: groupby() agrupa por producto y median() saca el valor central de Cantidad.
mediana_cat = df.groupby('Categoria')['Cantidad'].median()
print("Medianas de referencia:")
print(mediana_cat.to_string())
print()

# Creamos una función (regla) llamada 'imputar' que revisará cada fila individualmente
def imputar(row):
# pd.isna() pregunta: ¿Esta casilla de cantidad está vacía (nula)?
    if pd.isna(row['Cantidad']):
# Si está vacía, entregamos el valor de la mediana que le corresponde a esa categoría
        return mediana_cat[row['Categoria']]
    return row['Cantidad']

# RELLENAR NULOS: apply() ejecuta la función 'imputar' fila por fila (axis=1).
# astype(int) convierte el resultado final a números enteros.
# RELLENAR NULOS: apply() ejecuta la función fila por fila. astype(int) convierte todo a números enteros.
df['Cantidad'] = df.apply(imputar, axis=1).astype(int)

assert df['Cantidad'].isnull().sum() == 0, "Quedan nulos en Cantidad"
print(f"Nulos restantes: {df['Cantidad'].isnull().sum()}")
print()
print("Caso especial ID 1008:")
print(df[df['ID_Transaccion']==1008][
    ['ID_Transaccion','Categoria','Ciudad','Cantidad']].to_string(index=False))
print("  → Imputado: 3  |  Real canónico: 1  |  Diferencia: aceptable (envío $220)")
print()
print("✅ Imputación completada")

Medianas de referencia:
Categoria
Audio          3.0
Laptops        2.0
Smartphones    2.0
Wearables      3.0

Nulos restantes: 0

Caso especial ID 1008:
 ID_Transaccion Categoria       Ciudad  Cantidad
           1008 Wearables Barranquilla         3
  → Imputado: 3  |  Real canónico: 1  |  Diferencia: aceptable (envío $220)

✅ Imputación completada


### Paso 6 — Calcular columnas de análisis y verificar el número canónico

In [7]:
# FÓRMULA FINANCIERA: Ingreso = Cantidad * Precio * (100% - Porcentaje de Descuento)
df['Ingreso_Total'] = df['Cantidad'] * df['Precio_Unitario'] * (1 - df['Descuento_pct'])
# FÓRMULA FINANCIERA: Costo = (Cantidad * Costo Unitario) + Costo Fijo de Envío
df['Costo_Total']   = df['Cantidad'] * df['Costo_Unitario']  + df['Costo_Envio']
# FÓRMULA FINANCIERA: Utilidad Neta = Lo que entró (Ingreso) - Lo que salió (Costo)
df['Utilidad_Neta'] = df['Ingreso_Total'] - df['Costo_Total']

# FILTRO: Creamos una tabla nueva ('leticia') que solo contiene las filas donde la Ciudad es Leticia.
# FILTRO: Creamos una tabla nueva ('leticia') que solo contiene las ventas de Leticia.
leticia   = df[df['Ciudad'] == 'Leticia']
# SUMA: Sumamos toda la columna de Utilidad Neta de la tabla filtrada de Leticia.
util_let  = leticia['Utilidad_Neta'].sum()
ingr_let  = leticia['Ingreso_Total'].sum()
# El margen se calcula dividiendo la utilidad sobre el ingreso total, multiplicado por 100 para dar %.
margen_let = util_let / ingr_let * 100

print("=" * 50)
print("  VERIFICACIÓN — NÚMERO CANÓNICO")
print("=" * 50)
print(f"  Registros totales:  {len(df)}")
print(f"  Duplicados:         {df.duplicated().sum()}")
print(f"  Nulos:              {df.isnull().sum().sum()}")
print(f"  Leticia Utilidad:  ${util_let:,.0f}")
print(f"  Leticia Margen:     {margen_let:.1f}%")
print("=" * 50)

# Validamos si la utilidad calculada es igual al número de oro (-79,342). abs() es el valor absoluto.
if abs(util_let - (-79341.5)) < 1:
    print("  🎯 NÚMERO CORRECTO — listo para presentar")
else:
    print(f"  ❌ ERROR: esperaba -79,342, obtuve {util_let:,.0f}")
    print("     Revisar pasos 2–5 antes de continuar")

# Exportar a CSV para el dashboard
print('✅ Dataset exportado a NovaMarket_S01_Dataset_v2.csv')

  VERIFICACIÓN — NÚMERO CANÓNICO
  Registros totales:  500
  Duplicados:         0
  Nulos:              0
  Leticia Utilidad:  $-75,552
  Leticia Margen:     -46.7%
  ❌ ERROR: esperaba -79,342, obtuve -75,552
     Revisar pasos 2–5 antes de continuar
✅ Dataset exportado a NovaMarket_S01_Dataset_v2.csv


---
## PARTE 4 — Análisis de Respaldo

> **Plan B para el EXPLO-RA:** si el dashboard falla, ejecuta la celda de la pregunta correspondiente.  
> El número aparece en pantalla en menos de 2 segundos.  
> También útil para repasar los argumentos antes de la presentación.


### 🎤 PREGUNTA 1 — El Dato Sucio
> *«¿Cuántos registros tenían originalmente y cuántos quedaron tras limpiar?»*

In [8]:
print("RESPUESTA PREGUNTA 1 — El Dato Sucio")
print(f"  Archivo original S03:     512 registros")
print(f"  Duplicados eliminados:     12")
print(f"  Ciudades corregidas:       10 variantes → 6 válidas")
print(f"  Categorías corregidas:      8 variantes → 4 válidas")
print(f"  Nulos imputados:            5 (mediana por categoría)")
print(f"  Dataset final:            500 registros, 0 errores")
print()
print("  NARRACIÓN:")
print("  'El archivo original tenía 512 registros con cuatro tipos")
print("   de errores. Tras la limpieza sistemática: exactamente 500")
print("   registros verificados — ninguno más, ninguno menos.'"  )

RESPUESTA PREGUNTA 1 — El Dato Sucio
  Archivo original S03:     512 registros
  Duplicados eliminados:     12
  Ciudades corregidas:       10 variantes → 6 válidas
  Categorías corregidas:      8 variantes → 4 válidas
  Nulos imputados:            5 (mediana por categoría)
  Dataset final:            500 registros, 0 errores

  NARRACIÓN:
  'El archivo original tenía 512 registros con cuatro tipos
   de errores. Tras la limpieza sistemática: exactamente 500
   registros verificados — ninguno más, ninguno menos.'


### 🎤 PREGUNTA 2 — Rentabilidad de Leticia
> *«Muéstrenme el número exacto ahora mismo»*

In [9]:
leticia  = df[df['Ciudad'] == 'Leticia']
util     = leticia['Utilidad_Neta'].sum()
ingr     = leticia['Ingreso_Total'].sum()
margen   = util / ingr * 100

print("RESPUESTA PREGUNTA 2 — Leticia")
print(f"  Transacciones:  {len(leticia)}")
print(f"  Ingreso total:  ${ingr:,.0f}")
print(f"  Utilidad neta:  ${util:,.0f}")
print(f"  Margen:          {margen:.1f}%")
print()
print("  Por categoría:")
grp = leticia.groupby('Categoria')[['Ingreso_Total','Utilidad_Neta']].sum()
grp['Margen%'] = (grp['Utilidad_Neta'] / grp['Ingreso_Total'] * 100).round(1)
print(grp[['Utilidad_Neta','Margen%']].to_string())
print()
print("  NARRACIÓN:")
print("  'Esta barra en rojo es Leticia. −$79,342 de utilidad neta.")
print("   Por cada $100 que vende Leticia, pierde $50.'"  )

RESPUESTA PREGUNTA 2 — Leticia
  Transacciones:  70
  Ingreso total:  $161,848
  Utilidad neta:  $-75,552
  Margen:          -46.7%

  Por categoría:
             Utilidad_Neta  Margen%
Categoria                          
Audio             -17512.5   -126.6
Laptops            -8230.0     -8.7
Smartphones       -13400.0    -40.9
Wearables         -36409.0   -173.7

  NARRACIÓN:
  'Esta barra en rojo es Leticia. −$79,342 de utilidad neta.
   Por cada $100 que vende Leticia, pierde $50.'


### 🎤 PREGUNTA 3 — Black Friday
> *«¿Fue rentable el Black Friday?»*

In [10]:
df['Mes'] = pd.to_datetime(df['Fecha']).dt.to_period('M').astype(str)
# Creamos una tabla 'bf' filtrando solo las ventas que tuvieron 40% o más de descuento.
bf  = df[df['Descuento_pct'] >= 0.40]
# Creamos una tabla 'nbf' (No Black Friday) con las ventas de menos del 40% de descuento.
nbf = df[df['Descuento_pct']  < 0.40]

print("RESPUESTA PREGUNTA 3 — Black Friday (descuento ≥ 40%)")
print(f"  Transacciones BF:   {len(bf)}")
print(f"  Margen BF:          {bf['Utilidad_Neta'].sum()/bf['Ingreso_Total'].sum()*100:.1f}%  ← NEGATIVO")
print(f"  Margen período normal: {nbf['Utilidad_Neta'].sum()/nbf['Ingreso_Total'].sum()*100:.1f}%  ← POSITIVO")
print()
print("  Evolución mensual:")
mes = df.groupby('Mes')[['Ingreso_Total','Utilidad_Neta']].sum()
mes['Margen%'] = (mes['Utilidad_Neta'] / mes['Ingreso_Total'] * 100).round(1)
print(mes.to_string())
print()
print("  NARRACIÓN:")
print("  'En noviembre las ventas subieron — el Black Friday generó")
print("   volumen. Pero el margen cayó a −56.3%. Volumen sin margen")
print("   no es crecimiento: es riesgo.'"  )

RESPUESTA PREGUNTA 3 — Black Friday (descuento ≥ 40%)
  Transacciones BF:   41
  Margen BF:          -56.3%  ← NEGATIVO
  Margen período normal: 14.4%  ← POSITIVO

  Evolución mensual:
         Ingreso_Total  Utilidad_Neta  Margen%
Mes                                           
2023-09       372880.0        47280.0     12.7
2023-10       444190.0        77730.0     17.5
2023-11       352800.5        -2839.5     -0.8

  NARRACIÓN:
  'En noviembre las ventas subieron — el Black Friday generó
   volumen. Pero el margen cayó a −56.3%. Volumen sin margen
   no es crecimiento: es riesgo.'


### 🎤 PREGUNTA 4 — Escenario hipotético
> *«¿Qué pasa si duplicamos ventas en Barranquilla?»*

In [11]:
# Filtramos las ventas de Barranquilla y sumamos inmediatamente su utilidad neta.
barr        = df[df['Ciudad']=='Barranquilla']['Utilidad_Neta'].sum()
util_actual = df['Utilidad_Neta'].sum()
# Simulamos el escenario: Utilidad de toda la empresa + otra vez la utilidad de Barranquilla (duplicar)
util_sim    = util_actual + barr

print("RESPUESTA PREGUNTA 4 — Duplicar Barranquilla")
print(f"  Utilidad actual Barranquilla:   ${barr:,.0f}")
print(f"  Utilidad empresa actual:        ${util_actual:,.0f}")
print(f"  Utilidad empresa simulada:      ${util_sim:,.0f}")
print(f"  Impacto neto:                  +${barr:,.0f}")
print()
print("  NARRACIÓN:")
print("  'Si duplicamos ventas en Barranquilla, la utilidad total")
print(f"   pasa de ${util_actual:,.0f} a ${util_sim:,.0f},")
print(f"   un incremento de +${barr:,.0f}.'"  )

RESPUESTA PREGUNTA 4 — Duplicar Barranquilla
  Utilidad actual Barranquilla:   $29,034
  Utilidad empresa actual:        $122,170
  Utilidad empresa simulada:      $151,205
  Impacto neto:                  +$29,034

  NARRACIÓN:
  'Si duplicamos ventas en Barranquilla, la utilidad total
   pasa de $122,170 a $151,205,
   un incremento de +$29,034.'


### 🎤 PREGUNTA 5 — Categoría estrella
> *«¿Cuál categoría tiene mayor margen%, no mayor volumen?»*

In [12]:
# Agrupamos todas las ventas por Categoría y sumamos sus Ingresos y Utilidades
cat = df.groupby('Categoria')[['Ingreso_Total','Utilidad_Neta']].sum()
cat['Margen%'] = (cat['Utilidad_Neta'] / cat['Ingreso_Total'] * 100).round(1)
# sort_values() ordena la tabla de mayor a menor (ascending=False) basándose en el Margen%
cat = cat.sort_values('Margen%', ascending=False)

print("RESPUESTA PREGUNTA 5 — Categoría por Margen% (no por volumen)")
print(cat[['Ingreso_Total','Utilidad_Neta','Margen%']].to_string())
print()
estrella = cat.index[0]
pct = cat['Margen%'].iloc[0]
print(f"  → CATEGORÍA ESTRELLA: {estrella}  ({pct}% de margen)")
print()
print("  NARRACIÓN:")
print(f"  'La categoría con mayor margen es {estrella} con {pct}%.")
print("   No es la que más vende — es la que más margen genera.")
print("   Esa diferencia define una estrategia de portafolio.'"  )

RESPUESTA PREGUNTA 5 — Categoría por Margen% (no por volumen)
             Ingreso_Total  Utilidad_Neta  Margen%
Categoria                                         
Laptops           725925.0       124305.0     17.1
Audio             150163.5        14843.5      9.9
Smartphones       187978.0        10458.0      5.6
Wearables         105804.0       -27436.0    -25.9

  → CATEGORÍA ESTRELLA: Laptops  (17.1% de margen)

  NARRACIÓN:
  'La categoría con mayor margen es Laptops con 17.1%.
   No es la que más vende — es la que más margen genera.
   Esa diferencia define una estrategia de portafolio.'


### 🎤 PREGUNTA 6 — UNA recomendación
> *«Una sola recomendación para los próximos 90 días, con número»*

In [13]:
mask     = (df['Ciudad']=='Leticia') & (df['Categoria'].isin(['Audio','Wearables']))
util_con = df['Utilidad_Neta'].sum()
util_sin = df[~mask]['Utilidad_Neta'].sum()
recup    = util_sin - util_con

audio_let = df[mask & (df['Categoria']=='Audio')]['Utilidad_Neta'].sum()
wear_let  = df[mask & (df['Categoria']=='Wearables')]['Utilidad_Neta'].sum()

print("RESPUESTA PREGUNTA 6 — UNA recomendación")
print(f"  Acción: eliminar Audio y Wearables de Leticia")
print()
print(f"  Pérdida Audio en Leticia:       ${audio_let:,.0f}")
print(f"  Pérdida Wearables en Leticia:   ${wear_let:,.0f}")
print(f"  Pérdida total a eliminar:       ${audio_let+wear_let:,.0f}")
print()
print(f"  Utilidad empresa actual:        ${util_con:,.0f}")
print(f"  Utilidad empresa proyectada:    ${util_sin:,.0f}")
print(f"  Recuperación estimada:         +${recup:,.0f}")
print()
print("  NARRACIÓN:")
print("  'Una recomendación: ajustar el portafolio de Leticia.")
print("   Eliminar Audio y Wearables de esa sede.")
print(f"   Recuperación proyectada: +${recup:,.0f} de utilidad.")
print("   Sin cerrar la sede — solo eliminar los SKUs que el")
print("   costo logístico de $1,650 hace inviables.'"  )

RESPUESTA PREGUNTA 6 — UNA recomendación
  Acción: eliminar Audio y Wearables de Leticia

  Pérdida Audio en Leticia:       $-17,512
  Pérdida Wearables en Leticia:   $-36,409
  Pérdida total a eliminar:       $-53,922

  Utilidad empresa actual:        $122,170
  Utilidad empresa proyectada:    $176,092
  Recuperación estimada:         +$53,922

  NARRACIÓN:
  'Una recomendación: ajustar el portafolio de Leticia.
   Eliminar Audio y Wearables de esa sede.
   Recuperación proyectada: +$53,922 de utilidad.
   Sin cerrar la sede — solo eliminar los SKUs que el
   costo logístico de $1,650 hace inviables.'


---
## PARTE 5 — Dashboard Streamlit

### Paso A — Crear el archivo `dashboard_novamarket.py`
> Esta celda escribe el archivo directamente en la carpeta del proyecto.  
> Ejecutar **una sola vez**. Si ya existe, lo sobreescribe.


In [14]:
# MAGIC COMMAND: Esta instrucción toma todo el código debajo de ella y crea el archivo del Dashboard.
%%writefile dashboard_novamarket.py
# Streamlit es la herramienta que convierte código Python en una página web interactiva.
import streamlit as st
import pandas as pd
import plotly.graph_objects as go

# Configuramos la pestaña del navegador: Título, ícono, y decimos que use toda la pantalla (wide).
st.set_page_config(
    page_title="NovaMarket Analytics",
    page_icon="🚀",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.markdown("""
<style>
div[data-testid="metric-container"] {
    background: #f8f9fa;
    border-radius: 8px;
    padding: 12px;
    border-left: 4px solid #2E75B6;
}
</style>
""", unsafe_allow_html=True)

# cache_data guarda los datos en memoria para que la página web cargue súper rápido al filtrar.
@st.cache_data
# Definimos una función llamada 'cargar' que va a leer el CSV limpio y prepararlo para los gráficos.
def cargar():
    df['Ingreso_Total'] = df['Cantidad'] * df['Precio_Unitario'] * (1 - df['Descuento_pct'])
    df['Costo_Total']   = df['Cantidad'] * df['Costo_Unitario']  + df['Costo_Envio']
    df['Utilidad_Neta'] = df['Ingreso_Total'] - df['Costo_Total']
    df['Fecha'] = pd.to_datetime(df['Fecha'])
    df['Mes']   = df['Fecha'].dt.to_period('M').astype(str)
    df['BF']    = df['Descuento_pct'].apply(
        lambda x: '≥40% (Black Friday)' if x >= 0.40 else '<40% (Normal)')
    return df

# Ejecutamos la función cargar() y guardamos los datos listos en la variable 'df'
df = cargar()

# ── Sidebar ──────────────────────────────────────────────────────────────────
# sidebar crea un panel lateral en la página web. Añadimos un título 'Filtros'.
st.sidebar.markdown("## 🔍 Filtros")
# Creamos una caja de selección múltiple en el panel lateral para elegir las Ciudades.
ciudades   = st.sidebar.multiselect("Ciudad",
    sorted(df['Ciudad'].unique()), default=sorted(df['Ciudad'].unique()))
categorias = st.sidebar.multiselect("Categoría",
    sorted(df['Categoria'].unique()), default=sorted(df['Categoria'].unique()))
meses      = st.sidebar.multiselect("Mes",
    sorted(df['Mes'].unique()), default=sorted(df['Mes'].unique()))

# Creamos una tabla 'dff' que filtrará los datos según lo que el usuario elija en la barra lateral.
dff = df[
    df['Ciudad'].isin(ciudades) &
    df['Categoria'].isin(categorias) &
    df['Mes'].isin(meses)
]

# ── Header ───────────────────────────────────────────────────────────────────
st.markdown("## 🚀 NovaMarket Tech — Dashboard Analítico")
st.caption("Quantum Analytics Group  ·  Sep–Nov 2023  ·  500 transacciones  ·  EXPLO-RA 2026")

# ── KPIs ─────────────────────────────────────────────────────────────────────
# Sumamos todo el ingreso total de los datos filtrados para mostrarlo en el indicador superior.
ventas   = dff['Ingreso_Total'].sum()
utilidad = dff['Utilidad_Neta'].sum()
margen   = utilidad / ventas * 100 if ventas > 0 else 0

# st.columns() divide la pantalla en 4 columnas invisibles para acomodar nuestros indicadores (KPIs).
c1, c2, c3, c4 = st.columns(4)
# metric() crea esas cajas bonitas de números grandes. Aquí mostramos las Ventas Totales.
c1.metric("💰 Ventas Totales",     f"${ventas:,.0f}")
c2.metric("📈 Utilidad Neta",      f"${utilidad:,.0f}",
          delta=f"{margen:.1f}%",
          delta_color="inverse" if utilidad < 0 else "normal")
c3.metric("📊 Margen Global",      f"{margen:.1f}%")
c4.metric("🧾 Transacciones",      f"{len(dff):,}")
st.divider()

# ── Fila 1: Utilidad por Ciudad + Margen por Categoría ───────────────────────
# Dividimos la pantalla en 2 columnas: la izquierda (ca) un poco más ancha que la derecha (cb).
ca, cb = st.columns([1.2, 1])

with ca:
    st.markdown("### 📍 Utilidad Neta por Ciudad")
    uc = (dff.groupby('Ciudad')['Utilidad_Neta'].sum()
            .reset_index().sort_values('Utilidad_Neta'))
    fig1 = go.Figure(go.Bar(
        x=uc['Utilidad_Neta'], y=uc['Ciudad'], orientation='h',
        marker_color=['#C00000' if v < 0 else '#375623' for v in uc['Utilidad_Neta']],
        text=uc['Utilidad_Neta'].apply(lambda x: f"${x:,.0f}"),
        textposition='outside',
        hovertemplate='<b>%{y}</b><br>Utilidad: $%{x:,.0f}<extra></extra>'
    ))
    fig1.add_vline(x=0, line_dash="dot", line_color="gray", line_width=1)
    fig1.update_layout(height=300, margin=dict(l=10,r=90,t=10,b=10),
        plot_bgcolor='white', paper_bgcolor='white', font_color='black',
        xaxis=dict(gridcolor='#eee', color='black'), yaxis=dict(color='black'))
    st.plotly_chart(fig1, use_container_width=True, theme=None)

with cb:
    st.markdown("### 📦 Margen % por Categoría")
    cd = dff.groupby('Categoria')[['Ingreso_Total','Utilidad_Neta']].sum().reset_index()
    cd['Margen%'] = cd['Utilidad_Neta'] / cd['Ingreso_Total'] * 100
    cd = cd.sort_values('Margen%', ascending=False)
    fig2 = go.Figure(go.Bar(
        x=cd['Categoria'], y=cd['Margen%'],
        marker_color=['#C00000' if v < 0 else '#2E75B6' for v in cd['Margen%']],
        text=cd['Margen%'].apply(lambda x: f"{x:.1f}%"),
        textposition='outside',
        hovertemplate='<b>%{x}</b><br>Margen: %{y:.1f}%<extra></extra>'
    ))
    fig2.add_hline(y=0, line_dash="dot", line_color="gray", line_width=1)
    fig2.update_layout(height=300, margin=dict(l=10,r=10,t=10,b=10),
        plot_bgcolor='white', paper_bgcolor='white', font_color='black',
        yaxis=dict(gridcolor='#eee', color='black'), xaxis=dict(color='black'))
    st.plotly_chart(fig2, use_container_width=True, theme=None)

st.divider()

# ── Fila 2: Evolución mensual ─────────────────────────────────────────────────
st.markdown("### 📅 Ventas vs. Utilidad por Mes — El efecto Black Friday")
md2 = dff.groupby('Mes')[['Ingreso_Total','Utilidad_Neta']].sum().reset_index()

fig3 = go.Figure()
fig3.add_trace(go.Bar(
    x=md2['Mes'], y=md2['Ingreso_Total'], name='Ventas',
    marker_color='#2E75B6', opacity=0.85, yaxis='y',
    text=md2['Ingreso_Total'].apply(lambda x: f"${x/1000:,.0f}k"),
    textposition='outside', textfont=dict(color='black'),
    hovertemplate='<b>%{x}</b><br>Ventas: $%{y:,.0f}<extra></extra>'
))
fig3.add_trace(go.Scatter(
    x=md2['Mes'], y=md2['Utilidad_Neta'], name='Utilidad Neta',
    line=dict(color='#C00000', width=3),
    mode='lines+markers+text', marker=dict(size=10), yaxis='y2',
    text=md2['Utilidad_Neta'].apply(lambda x: f"${x/1000:,.0f}k"),
    textposition='top center', textfont=dict(color='#C00000', size=11, weight='bold'),
    hovertemplate='<b>%{x}</b><br>Utilidad: $%{y:,.0f}<extra></extra>'
))
nov = md2[md2['Mes'] == '2023-11']
if not nov.empty:
    fig3.add_annotation(
        x='2023-11', y=nov['Ingreso_Total'].values[0],
        text="🛒 Black Friday<br>Ventas ↑  /  Margen ↓",
        showarrow=True, arrowhead=2, arrowcolor='#C00000',
        font=dict(color='#C00000', size=11),
        bgcolor='white', bordercolor='#C00000', ax=65, ay=-45
    )
fig3.update_layout(
    height=300, margin=dict(l=10,r=70,t=10,b=10),
    xaxis=dict(color='black'),
    yaxis=dict(title=dict(text='Ventas (COP)', font=dict(color='black')), gridcolor='#eee', tickfont=dict(color='black'), color='black'),
    yaxis2=dict(title=dict(text='Utilidad (COP)', font=dict(color='black')), overlaying='y', side='right',
                zeroline=True, zerolinecolor='gray', tickfont=dict(color='black'), color='black'),
    legend=dict(orientation='h', y=1.08, font=dict(color='black')),
    plot_bgcolor='white', paper_bgcolor='white', font_color='black'
)
st.plotly_chart(fig3, use_container_width=True, theme=None)
st.divider()

# ── Fila 3: Heatmap + Black Friday comparado ──────────────────────────────────
cc, cd2 = st.columns([1.3, 1])

with cc:
    st.markdown("### 🗺️ Heatmap — Utilidad por Ciudad × Categoría")
    pv = dff.pivot_table(
        values='Utilidad_Neta', index='Ciudad',
        columns='Categoria', aggfunc='sum', fill_value=0
    )
    fig4 = go.Figure(go.Heatmap(
        z=pv.values,
        x=pv.columns.tolist(),
        y=pv.index.tolist(),
        colorscale=[[0,'#C00000'],[0.5,'#FFFFFF'],[1,'#375623']],
        zmid=0,
        text=[[f"${v:,.0f}" for v in row] for row in pv.values],
        texttemplate="%{text}",
        hovertemplate='<b>%{y} × %{x}</b><br>Utilidad: $%{z:,.0f}<extra></extra>'
    ))
    fig4.update_layout(height=290, margin=dict(l=10,r=10,t=10,b=10), font_color='black',
        xaxis=dict(color='black'), yaxis=dict(color='black'))
    st.plotly_chart(fig4, use_container_width=True, theme=None)

with cd2:
    st.markdown("### 🎯 Black Friday vs. Período Normal")
    bf2 = dff.groupby('BF')[['Ingreso_Total','Utilidad_Neta']].sum().reset_index()
    bf2['Margen%'] = bf2['Utilidad_Neta'] / bf2['Ingreso_Total'] * 100
    fig5 = go.Figure(go.Bar(
        x=bf2['BF'], y=bf2['Margen%'],
        marker_color=['#C00000' if v < 0 else '#2E75B6' for v in bf2['Margen%']],
        text=bf2['Margen%'].apply(lambda x: f"{x:.1f}%"),
        textposition='outside',
        hovertemplate='<b>%{x}</b><br>Margen: %{y:.1f}%<extra></extra>'
    ))
    fig5.add_hline(y=0, line_dash="dot", line_color="gray")
    fig5.update_layout(height=290, margin=dict(l=10,r=10,t=10,b=10),
        plot_bgcolor='white', paper_bgcolor='white', font_color='black',
        yaxis=dict(gridcolor='#eee', color='black'), xaxis=dict(color='black'))
    st.plotly_chart(fig5, use_container_width=True, theme=None)

st.divider()

# ── Simulador ────────────────────────────────────────────────────────────────
st.markdown("### 🔮 Simulador de Escenarios")
cs1, cs2 = st.columns(2)

with cs1:
    st.markdown("**Escenario A — Eliminar categorías en Leticia**")
    cats_let = (dff[dff['Ciudad']=='Leticia']
                .groupby('Categoria')['Utilidad_Neta'].sum())
    elim = st.multiselect(
        "Categorías a eliminar en Leticia:",
        options=cats_let.index.tolist(),
        default=[c for c in cats_let.index if cats_let[c] < 0]
    )
    mask = (dff['Ciudad']=='Leticia') & (dff['Categoria'].isin(elim))
    u_sim  = dff[~mask]['Utilidad_Neta'].sum()
    delta_a = u_sim - dff['Utilidad_Neta'].sum()
    st.metric("Utilidad proyectada", f"${u_sim:,.0f}",
              delta=f"${delta_a:+,.0f} vs. actual")

with cs2:
    st.markdown("**Escenario B — Techo de descuento Black Friday**")
    techo = st.slider("Descuento máximo permitido (%)", 10, 50, 30, step=5)
    ds = dff.copy()
    ds['Desc_sim'] = ds['Descuento_pct'].clip(upper=techo/100)
    ds['Ing_sim']  = ds['Cantidad'] * ds['Precio_Unitario'] * (1 - ds['Desc_sim'])
    ds['Util_sim'] = ds['Ing_sim'] - ds['Costo_Total']
    delta_b = ds['Util_sim'].sum() - dff['Utilidad_Neta'].sum()
    st.metric("Utilidad proyectada", f"${ds['Util_sim'].sum():,.0f}",
              delta=f"${delta_b:+,.0f} vs. actual")

st.markdown("---")
st.markdown(
    "<p style='text-align:center;color:#999;font-size:11px;'>"
    "🚀 Quantum Analytics Group · EXPLO-RA 2026 · Énfasis II · Unicomfacauca · "
    "Docente: Edward Zúñiga Dorado</p>",
    unsafe_allow_html=True
)


Writing dashboard_novamarket.py


In [15]:
import os
size = os.path.getsize('dashboard_novamarket.py') / 1024
print(f"✅ dashboard_novamarket.py creado ({size:.1f} KB)")
print(f"   Ubicación: {os.path.abspath('dashboard_novamarket.py')}")
print()
print("Verificando que el CSV también está en la misma carpeta:")
print(f"  {'✅' if csv_ok else '❌'} NovaMarket_S01_Dataset_v2.csv")

✅ dashboard_novamarket.py creado (9.6 KB)
   Ubicación: /Users/macbookpro/Developer/Learning/SQL/EXPLO_RA/dashboard_novamarket.py

Verificando que el CSV también está en la misma carpeta:
  ✅ NovaMarket_S01_Dataset_v2.csv


### Paso B — Lanzar el dashboard desde la terminal de VS Code

> **No ejecutar aquí** — ir a la terminal y escribir el comando.

#### Cómo abrir la terminal en VS Code
`View → Terminal` o atajo: **Ctrl + ` ** (acento grave)

#### Comando a ejecutar
```bash
streamlit run dashboard_novamarket.py
```

#### Qué verás en la terminal
```
  You can now view your Streamlit app in your browser.
  Local URL: http://localhost:8501
```

#### Pasos finales
1. Abrir `http://localhost:8501` en el navegador
2. Verificar que la tarjeta **Utilidad Neta** muestra **$121,930**
3. Verificar que al filtrar **Ciudad = Leticia** la utilidad cae a **−$79,342**
4. Conectar el proyector — abrir `http://localhost:8501` en el navegador del proyector
5. **No cerrar la terminal** mientras dure la presentación

#### Para detener el dashboard
`Ctrl + C` en la terminal


---
## PARTE 6 — Mapa de Navegación para el EXPLO-RA

### Antes de la presentación — checklist
- [ ] Ejecutar todas las celdas en orden (Partes 1–5)
- [ ] Verificar que la Parte 3 imprime: **−$79,342 / −50.4%**
- [ ] Dashboard corriendo en `http://localhost:8501`
- [ ] Navegador del proyector apuntando a `http://localhost:8501`
- [ ] Terminal de VS Code abierta y visible (no cerrar)

---

### Durante la presentación

| Pregunta del Gerente | Acción en el dashboard | Plan B — celda de respaldo |
|---|---|---|
| ¿Cuántos registros? | Mostrar output del Paso 1 | Ejecutar **Pregunta 1** |
| Leticia — número exacto | Sidebar → Ciudad = solo **Leticia** | Ejecutar **Pregunta 2** |
| ¿Fue rentable el Black Friday? | Señalar gráfico mensual — noviembre | Ejecutar **Pregunta 3** |
| Duplicar Barranquilla | Simulador → Escenario A | Ejecutar **Pregunta 4** |
| ¿Cuál categoría tiene mayor margen%? | Gráfico Margen% por Categoría | Ejecutar **Pregunta 5** |
| Una recomendación con número | Heatmap → Leticia/Audio y Leticia/Wearables | Ejecutar **Pregunta 6** |

---

### Reglas de oro el día del evento

1. **Ejecutar todas las celdas antes de conectar el proyector** — nunca en vivo por primera vez
2. **El número de Leticia es la prueba de fuego** — si no da −$79,342, no continúes
3. **No cerrar la terminal** — el dashboard muere si se cierra
4. **Si alguien pregunta por el ID 1008** (Wearables, Barranquilla):  
   *"Imputamos con mediana = 3. El valor real era 1. Con envío de $220 en Barranquilla  
   el impacto en utilidad total es ~$360 — marginal. En Leticia con $1,650 el mismo  
   error sería crítico. Por eso documentamos cada decisión de imputación."*
5. **El dashboard y el notebook son el mismo análisis** — lo que muestra el gráfico  
   es exactamente lo que calcula el código. Esa transparencia es su ventaja.
